In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/cleaned_eda.csv")
df_fe = df.copy()

In [3]:
# Age cleanup
df_fe['Delivery_person_Age'] = pd.to_numeric(df_fe['Delivery_person_Age'], errors='coerce')
df_fe = df_fe[(df_fe['Delivery_person_Age'] >= 18) & (df_fe['Delivery_person_Age'] <= 60)]

# Ratings cleanup
df_fe['Delivery_person_Ratings'] = pd.to_numeric(df_fe['Delivery_person_Ratings'], errors='coerce')
df_fe = df_fe[(df_fe['Delivery_person_Ratings'] >= 1) & (df_fe['Delivery_person_Ratings'] <= 5)]

# Vehicle condition (ensure numeric)
df_fe['Vehicle_condition'] = pd.to_numeric(df_fe['Vehicle_condition'], errors='coerce')

In [4]:
# Clean target
df_fe['Time_taken(min)'] = pd.to_numeric(df_fe['Time_taken(min)'], errors='coerce')

# Drop null target rows
df_fe = df_fe.dropna(subset=['Time_taken(min)'])

In [5]:
df_fe['distance_bucket'] = pd.cut(
    df_fe['delivery_distance_km'],
    bins=[0, 2, 5, 10, 20, 50],
    labels=['very_short', 'short', 'medium', 'long', 'very_long']
)

In [6]:
df_fe['rating_distance_interaction'] = (
    df_fe['Delivery_person_Ratings'] * df_fe['delivery_distance_km']
)

In [7]:
traffic_map = {
    'Low': 1,
    'Medium': 2,
    'High': 3,
    'Jam': 4
}

df_fe['traffic_encoded'] = df_fe['Road_traffic_density'].map(traffic_map)

In [8]:
weather_map = {
    'conditions Sunny': 1,
    'conditions Cloudy': 2,
    'conditions Fog': 3,
    'conditions Stormy': 4,
    'conditions Sandstorms': 4,
    'conditions Windy': 3
}

df_fe['weather_encoded'] = df_fe['Weatherconditions'].map(weather_map)

In [9]:
df_fe['distance_rush_interaction'] = (
    df_fe['delivery_distance_km'] * df_fe['is_rush_hour']
)

In [10]:
df_fe['multi_delivery_bucket'] = df_fe['multiple_deliveries'].apply(
    lambda x: 'single' if x == 0 else ('double' if x == 1 else 'multiple')
)

In [11]:
df_fe['estimated_speed'] = (
    df_fe['delivery_distance_km'] / df_fe['Time_taken(min)']
)

In [12]:
df_fe = df_fe.drop(columns=[
    'Weatherconditions',
    'Road_traffic_density'
])

In [13]:
df_model = df_fe.drop(columns=['estimated_speed'])

### encoding

In [14]:
categorical_cols = [
    'Type_of_order',
    'Type_of_vehicle_merged',
    'City',
    'distance_bucket',
    'multi_delivery_bucket'
]

df_model = pd.get_dummies(df_model, columns=categorical_cols, drop_first=True)

In [18]:
df_model.columns

Index(['Delivery_person_Age', 'Delivery_person_Ratings', 'Vehicle_condition',
       'Type_of_vehicle', 'multiple_deliveries', 'Time_taken(min)',
       'delivery_distance_km', 'order_hour', 'order_day_of_week',
       'order_month', 'is_weekend', 'prep_time_minutes', 'is_rush_hour',
       'rating_distance_interaction', 'traffic_encoded', 'weather_encoded',
       'distance_rush_interaction', 'Type_of_order_Drinks ',
       'Type_of_order_Meal ', 'Type_of_order_Snack ',
       'Type_of_vehicle_merged_scooter ', 'City_Semi-Urban', 'City_Urban',
       'distance_bucket_short', 'distance_bucket_medium',
       'distance_bucket_long', 'distance_bucket_very_long',
       'multi_delivery_bucket_multiple', 'multi_delivery_bucket_single'],
      dtype='object')

In [19]:
df_model = df_model.drop(columns=['prep_time_minutes'])

In [20]:
assert 'estimated_speed' not in df_model.columns

In [21]:
df_model.columns = df_model.columns.str.strip()

In [22]:
df_model = df_model.drop(columns=[
    'Type_of_vehicle'
])

In [23]:
df_model.columns

Index(['Delivery_person_Age', 'Delivery_person_Ratings', 'Vehicle_condition',
       'multiple_deliveries', 'Time_taken(min)', 'delivery_distance_km',
       'order_hour', 'order_day_of_week', 'order_month', 'is_weekend',
       'is_rush_hour', 'rating_distance_interaction', 'traffic_encoded',
       'weather_encoded', 'distance_rush_interaction', 'Type_of_order_Drinks',
       'Type_of_order_Meal', 'Type_of_order_Snack',
       'Type_of_vehicle_merged_scooter', 'City_Semi-Urban', 'City_Urban',
       'distance_bucket_short', 'distance_bucket_medium',
       'distance_bucket_long', 'distance_bucket_very_long',
       'multi_delivery_bucket_multiple', 'multi_delivery_bucket_single'],
      dtype='object')

In [24]:
import numpy as np

# Hour cyclical encoding
df_model['order_hour_sin'] = np.sin(2 * np.pi * df_model['order_hour'] / 24)
df_model['order_hour_cos'] = np.cos(2 * np.pi * df_model['order_hour'] / 24)

# Day of week cyclical
df_model['dow_sin'] = np.sin(2 * np.pi * df_model['order_day_of_week'] / 7)
df_model['dow_cos'] = np.cos(2 * np.pi * df_model['order_day_of_week'] / 7)

# Drop original columns
df_model = df_model.drop(columns=['order_hour', 'order_day_of_week'])

In [27]:
df_model.columns

Index(['Delivery_person_Age', 'Delivery_person_Ratings', 'Vehicle_condition',
       'multiple_deliveries', 'Time_taken(min)', 'delivery_distance_km',
       'order_month', 'is_weekend', 'is_rush_hour',
       'rating_distance_interaction', 'traffic_encoded', 'weather_encoded',
       'distance_rush_interaction', 'Type_of_order_Drinks',
       'Type_of_order_Meal', 'Type_of_order_Snack',
       'Type_of_vehicle_merged_scooter', 'City_Semi-Urban', 'City_Urban',
       'distance_bucket_short', 'distance_bucket_medium',
       'distance_bucket_long', 'distance_bucket_very_long',
       'multi_delivery_bucket_multiple', 'multi_delivery_bucket_single',
       'order_hour_sin', 'order_hour_cos', 'dow_sin', 'dow_cos'],
      dtype='object')

In [28]:
df_model = df_model.drop(columns=[
    'distance_bucket_short',
    'distance_bucket_medium',
    'distance_bucket_long',
    'distance_bucket_very_long'
])

In [29]:
df_model = df_model.drop(columns=['order_month'])

In [30]:
df_model['multiple_deliveries'] = df_model['multiple_deliveries'].astype(int)

df_model = df_model.drop(columns=[
    'multi_delivery_bucket_multiple',
    'multi_delivery_bucket_single'
])

In [31]:
df_model.columns

Index(['Delivery_person_Age', 'Delivery_person_Ratings', 'Vehicle_condition',
       'multiple_deliveries', 'Time_taken(min)', 'delivery_distance_km',
       'is_weekend', 'is_rush_hour', 'rating_distance_interaction',
       'traffic_encoded', 'weather_encoded', 'distance_rush_interaction',
       'Type_of_order_Drinks', 'Type_of_order_Meal', 'Type_of_order_Snack',
       'Type_of_vehicle_merged_scooter', 'City_Semi-Urban', 'City_Urban',
       'order_hour_sin', 'order_hour_cos', 'dow_sin', 'dow_cos'],
      dtype='object')

### final splitting

In [33]:
X = df_model.drop(columns=['Time_taken(min)'])
y = df_model['Time_taken(min)']

In [34]:
X.describe().T

,count,mean,std,min,25%,50%,75%,max
Delivery_person_Age,38064.0,29.608948,5.761658,20.000000,25.000000,30.000000,35.000000,39.000000
Delivery_person_Ratings,38064.0,4.631870,0.316950,2.500000,4.500000,4.700000,4.900000,5.000000
Vehicle_condition,38064.0,0.995271,0.817619,0.000000,0.000000,1.000000,2.000000,2.000000
multiple_deliveries,38064.0,0.748818,0.572731,0.000000,0.000000,1.000000,1.000000,3.000000
delivery_distance_km,38064.0,28.384251,311.431814,1.465067,4.658182,9.220373,13.682165,6884.726399
is_weekend,38064.0,0.274249,0.446141,0.000000,0.000000,0.000000,1.000000,1.000000
is_rush_hour,38064.0,0.376261,0.484453,0.000000,0.000000,0.000000,1.000000,1.000000
rating_distance_interaction,38064.0,131.189651,1444.127650,5.214232,22.480031,43.314732,62.877702,34423.631995
traffic_encoded,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weather_encoded,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# No food delivery is:

# 300 km
# 6000 km
# IMPACT
# Model will overfit to extreme values
# Feature interactions (rating_distance_interaction) also corrupted
# RMSE will explode


# Clip outliers
df_model = df_model[df_model['delivery_distance_km'] < 50]